# Laboratorio 9 — Redes Neuronales Artificiales (RNA)
## CC3074 Minería de Datos · SmartStay Advisors · Semestre I 2026

**Objetivo:** Construir modelos de RNA para (1) clasificar el precio de listings Airbnb en categorías Barato / Medio / Caro y (2) predecir el precio numérico, comparando ambos tipos contra los algoritmos de laboratorios anteriores.

## 0. Carga de librerías

In [65]:
# ─── Librerías ───────────────────────────────────────────────────────────────
import warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline

# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Estilo visual ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('✓ Librerías cargadas')

✓ Librerías cargadas


## 1. Carga y preparación del dataset

Se carga `listings.RData` y se replica **exactamente** el mismo preprocesamiento y la misma partición 80/20 usados en los laboratorios anteriores para que la comparación sea válida.

In [66]:
# ─── 1.1  Carga CORRECTA del dataset ───────────────────────────────────────
import pyreadr

result = pyreadr.read_r('../data/listings.RData')

# Ver qué objetos hay dentro
print(result.keys())

# Tomar el objeto correcto
df = list(result.values())[0]

print('Shape inicial:', df.shape)
print(df.head())

odict_keys(['listings'])
Shape inicial: (171748, 80)
        id                         listing_url     scrape_id last_scraped  \
0   5456.0   https://www.airbnb.com/rooms/5456  2.025092e+13   2025-09-17   
1   6448.0   https://www.airbnb.com/rooms/6448  2.025092e+13   2025-09-17   
2   8502.0   https://www.airbnb.com/rooms/8502  2.025092e+13   2025-09-17   
3  13035.0  https://www.airbnb.com/rooms/13035  2.025092e+13   2025-09-17   
4  22828.0  https://www.airbnb.com/rooms/22828  2.025092e+13   2025-09-16   

        source                                               name  \
0  city scrape          Walk to 6th, Rainey St and Convention Ctr   
1  city scrape  Secluded Studio @ Zilker - King Bed, Bright & ...   
2  city scrape                            Woodland Studio Lodging   
3  city scrape      Historic house in highly walkable East Austin   
4  city scrape                 Garage Apartment central SE Austin   

                                         description  \
0  Great cent

In [67]:
# ─── 1.2  Verificar columnas categóricas ─────────────────────────────────────

cols = ['price', 'room_type', 'neighbourhood_cleansed', 'property_type']

for c in cols:
    if c in df.columns:
        print(f'✓ {c}')
    else:
        print(f'✗ {c} NO encontrada')

✓ price
✓ room_type
✓ neighbourhood_cleansed
✓ property_type


In [68]:
# ─── 1.3  Revisar tipos de datos ────────────────────────────────────────────
print(df.dtypes)

id                                              float64
listing_url                                         str
scrape_id                                       float64
last_scraped                                        str
source                                              str
                                                 ...   
calculated_host_listings_count_entire_homes       int32
calculated_host_listings_count_private_rooms      int32
calculated_host_listings_count_shared_rooms       int32
reviews_per_month                               float64
city                                                str
Length: 80, dtype: object


In [69]:
# ─── 1.4  Verificar columnas del dataset ─────────────────────────────────────
print(f'Número de columnas: {df.shape[1]}')
print(df.columns.tolist())

Número de columnas: 80
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'avail

In [70]:
# ─── 1.5  Seleccionar variables numéricas ────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f'Columnas numéricas: {len(num_cols)}')
print(num_cols[:10])

Columnas numéricas: 33
['id', 'scrape_id', 'host_id', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'minimum_nights', 'maximum_nights', 'minimum_nights_avg_ntm']


In [71]:
# ─── 1.6  Dimensiones del dataset ────────────────────────────────────────────
print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]:,}')

Filas: 171,748
Columnas: 80


In [72]:
# ─── 1.7  Seleccionar columnas de interés ────────────────────────────────────

target_cols = [
    'latitude','longitude','accommodates','bedrooms','beds','price',
    'minimum_nights','number_of_reviews','review_scores_rating','availability_365'
]

df_selected = df[target_cols].copy()

print(df_selected.head())

   latitude  longitude  accommodates bedrooms beds    price  minimum_nights  \
0  30.26057  -97.73441             3        1    2   $97.00               2   
1  30.26034  -97.76487             2        1    2  $160.00               3   
2  30.23466  -97.73682             2        1    1   $38.00               4   
3  30.26098  -97.73072             3        2    2  $145.00              15   
4  30.23614  -97.73225             2        1    1   $58.00              30   

   number_of_reviews  review_scores_rating  availability_365  
0                708                  4.85               328  
1                339                  4.97               316  
2                 54                  4.57                88  
3                 19                  5.00               321  
4                 56                  4.95               211  


In [73]:
# ─── 1.8  DataFrame base listo ──────────────────────────────────────────────
print(f'DataFrame: {df.shape}')
print(df.head(3))

print('\nEstadísticas básicas:')
print(df.describe().round(2))

DataFrame: (171748, 80)
       id                        listing_url     scrape_id last_scraped  \
0  5456.0  https://www.airbnb.com/rooms/5456  2.025092e+13   2025-09-17   
1  6448.0  https://www.airbnb.com/rooms/6448  2.025092e+13   2025-09-17   
2  8502.0  https://www.airbnb.com/rooms/8502  2.025092e+13   2025-09-17   

        source                                               name  \
0  city scrape          Walk to 6th, Rainey St and Convention Ctr   
1  city scrape  Secluded Studio @ Zilker - King Bed, Bright & ...   
2  city scrape                            Woodland Studio Lodging   

                                         description  \
0  Great central  location for walking to Convent...   
1  Clean, private space with everything you need ...   
2  Studio rental on lower level of home located i...   

                               neighborhood_overview  \
0  My neighborhood is ideally located if you want...   
1  The neighborhood is fun and funky (but quiet)!...   
2    

In [74]:
# ─── 1.9  Columnas enteras ──────────────────────────────────────────────────
int_cols = df.select_dtypes(include=['int64']).columns.tolist()

print(f'Columnas enteras: {len(int_cols)}')
print(int_cols)

Columnas enteras: 0
[]


In [75]:
# ─── 1.10  Columnas booleanas ───────────────────────────────────────────────
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

print(f'Columnas booleanas: {len(bool_cols)}')
print(bool_cols)

Columnas booleanas: 0
[]


In [76]:
# ─── 1.11  Columnas categóricas ─────────────────────────────────────────────
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'Columnas categóricas: {len(cat_cols)}')
print(cat_cols[:10])

# Ejemplo: valores únicos de room_type
if 'room_type' in df.columns:
    print('\nroom_type valores únicos:')
    print(df['room_type'].unique())

Columnas categóricas: 47
['listing_url', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_url', 'host_name', 'host_since']

room_type valores únicos:
<StringArray>
['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']
Length: 4, dtype: str


In [77]:
# ─── 1.12  Revisar cardinalidad de variables categóricas ─────────────────────
for col in df.select_dtypes(include=['object']).columns:
    print(f'{col}: {df[col].nunique()} valores únicos')

listing_url: 171748 valores únicos
last_scraped: 12 valores únicos
source: 2 valores únicos
name: 163402 valores únicos
description: 142952 valores únicos
neighborhood_overview: 61258 valores únicos
picture_url: 167625 valores únicos
host_url: 78818 valores únicos
host_name: 22307 valores únicos
host_since: 5975 valores únicos
host_location: 2871 valores únicos
host_about: 41618 valores únicos
host_response_time: 6 valores únicos
host_response_rate: 85 valores únicos
host_acceptance_rate: 103 valores únicos
host_is_superhost: 3 valores únicos
host_thumbnail_url: 76194 valores únicos
host_picture_url: 76221 valores únicos
host_neighbourhood: 2832 valores únicos
host_listings_count: 394 valores únicos
host_total_listings_count: 481 valores únicos
host_verifications: 8 valores únicos
host_has_profile_pic: 3 valores únicos
host_identity_verified: 3 valores únicos
neighbourhood: 3 valores únicos
neighbourhood_cleansed: 855 valores únicos
neighbourhood_group_cleansed: 17 valores únicos
prope

In [78]:
# ─── 1.13  Codificación de variables categóricas ─────────────────────────────
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Ejemplo con room_type
if 'room_type' in df.columns:
    df['room_type_code'] = le.fit_transform(df['room_type'])
    print(df[['room_type', 'room_type_code']].head())

# Booleanos (si vienen como texto)
for col in ['host_is_superhost', 'instant_bookable']:
    if col in df.columns:
        df[col] = df[col].map({'t': 1, 'f': 0, 'TRUE': 1, 'FALSE': 0}).astype('float')

         room_type  room_type_code
0  Entire home/apt               0
1  Entire home/apt               0
2  Entire home/apt               0
3  Entire home/apt               0
4  Entire home/apt               0


In [79]:
# ─── 1.14  Columnas de conteo (ejemplo) ─────────────────────────────────────
count_cols = [
    'host_listings_count',
    'calculated_host_listings_count',
    'number_of_reviews'
]

for col in count_cols:
    if col in df.columns:
        print(f'\n{col}:')
        print(df[col].describe().round(2))


host_listings_count:
count     170872
unique       394
top            1
freq       51744
Name: host_listings_count, dtype: int64

calculated_host_listings_count:
count    171748.00
mean         45.88
std         135.16
min           1.00
25%           1.00
50%           3.00
75%          21.00
max        1189.00
Name: calculated_host_listings_count, dtype: float64

number_of_reviews:
count    171748.00
mean         44.00
std          89.96
min           0.00
25%           1.00
50%           8.00
75%          46.00
max        3920.00
Name: number_of_reviews, dtype: float64


In [80]:
# ─── 1.15  Verificar columnas clave ─────────────────────────────────────────
cols_check = [
    'host_listings_count',
    'calculated_host_listings_count'
]

for col in cols_check:
    if col in df.columns:
        print(f'✓ {col} disponible')
    else:
        print(f'✗ {col} no encontrada')

print(f'\nDataFrame final: {df.shape}')

✓ host_listings_count disponible
✓ calculated_host_listings_count disponible

DataFrame final: (171748, 81)


In [81]:
# ─── 1.16  Limpieza y filtrado ──────────────────────────────────────────────
print(f'Filas antes de limpieza: {len(df):,}')

# Asegurar que price sea numérico
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Eliminar filas con price NA o <= 0
df = df[df['price'].notna() & (df['price'] > 0)].copy()

# Eliminar outliers extremos de precio (>= percentil 99)
p99 = df['price'].quantile(0.99)
df = df[df['price'] <= p99].copy()

# Eliminar filas sin coordenadas válidas
df = df[df['latitude'].notna() & df['longitude'].notna()].copy()

print(f'Filas después de limpieza: {len(df):,}')
print(f'Precio: min=${df.price.min():.0f}, max=${df.price.max():.0f}, median=${df.price.median():.0f}')

Filas antes de limpieza: 171,748
Filas después de limpieza: 0
Precio: min=$nan, max=$nan, median=$nan


In [82]:
# ─── 1.17  Crear variable categórica de precio (misma que labs anteriores) ────
# Tertiles: Barato / Medio / Caro
q33 = df['price'].quantile(1/3)
q67 = df['price'].quantile(2/3)

def categorize_price(p):
    if p <= q33: return 'Barato'
    if p <= q67: return 'Medio'
    return 'Caro'

df['price_category'] = df['price'].apply(categorize_price)

print(f'Umbrales de categoría:')
print(f'  Barato: price <= ${q33:.0f}')
print(f'  Medio:  ${q33:.0f} < price <= ${q67:.0f}')
print(f'  Caro:   price > ${q67:.0f}')
print(f'\nDistribución:')
print(df['price_category'].value_counts())

Umbrales de categoría:
  Barato: price <= $nan
  Medio:  $nan < price <= $nan
  Caro:   price > $nan

Distribución:
Series([], Name: count, dtype: int64)


In [83]:
# ─── 1.18  Seleccionar features para modelado ─────────────────────────────────
NUMERIC_FEATURES = [
    'latitude', 'longitude', 'accommodates', 'bedrooms', 'beds',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'availability_365'
]

# Agregar room_type_code si existe
if 'room_type_code' in df.columns:
    NUMERIC_FEATURES.append('room_type_code')
if 'host_is_superhost' in df.columns:
    NUMERIC_FEATURES.append('host_is_superhost')
if 'instant_bookable' in df.columns:
    NUMERIC_FEATURES.append('instant_bookable')
if 'host_listings_count' in df.columns:
    NUMERIC_FEATURES.append('host_listings_count')

# Filtrar solo las que existen en df
FEATURES = [f for f in NUMERIC_FEATURES if f in df.columns]
print(f'Features disponibles ({len(FEATURES)}): {FEATURES}')

# Subconjunto sin NA en features
df_model = df[FEATURES + ['price', 'price_category']].dropna().copy()
print(f'\nFilas para modelado: {len(df_model):,}')

Features disponibles (13): ['latitude', 'longitude', 'accommodates', 'bedrooms', 'beds', 'minimum_nights', 'number_of_reviews', 'review_scores_rating', 'availability_365', 'room_type_code', 'host_is_superhost', 'instant_bookable', 'host_listings_count']

Filas para modelado: 0


In [86]:
# ─── 1.16  Limpieza correcta ───────────────────────────────────────────────
print(f'Filas antes de limpieza: {len(df):,}')

# Limpiar price (viene como "$1,234")
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Mantener solo filas válidas
df = df[df['price'].notna() & (df['price'] > 0)].copy()

print(f'Filas después de limpieza: {len(df):,}')
print(df['price'].describe())

Filas antes de limpieza: 0
Filas después de limpieza: 0
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: price, dtype: float64


---
## 2. RNA para Clasificación de Precio

Se construyen **dos modelos** con diferentes topologías y funciones de activación.

### 2.1 Modelo 1 — Red "Profunda" con activación ReLU

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (128, 64, 32) — 3 capas |
| Activación | `relu` |
| Solver | `adam` |
| Regularización (alpha) | 0.001 |
| Max iteraciones | 500 |

In [85]:
# ─── Modelo 1: Profundo + ReLU ────────────────────────────────────────────────
print('Entrenando Modelo 1 (128-64-32, relu, adam)...')
t0 = time.time()

mlp1_clf = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED,
    verbose=False
)
mlp1_clf.fit(X_train_s, y_cat_train)
t1 = time.time()

y1_pred = mlp1_clf.predict(X_test_s)
y1_pred_train = mlp1_clf.predict(X_train_s)

acc1_test  = accuracy_score(y_cat_test, y1_pred)
acc1_train = accuracy_score(y_cat_train, y1_pred_train)
time1 = t1 - t0

print(f'\n✓ Modelo 1 entrenado en {time1:.1f}s')
print(f'  Iteraciones: {mlp1_clf.n_iter_}')
print(f'  Accuracy TRAIN: {acc1_train:.4f}')
print(f'  Accuracy TEST:  {acc1_test:.4f}')
print(f'  Sobreajuste:    {acc1_train - acc1_test:.4f}')

Entrenando Modelo 1 (128-64-32, relu, adam)...


NameError: name 'X_train_s' is not defined

### 2.2 Modelo 2 — Red "Ancha" con activación Tanh

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (256, 128) — 2 capas |
| Activación | `tanh` |
| Solver | `sgd` con momentum |
| Regularización (alpha) | 0.0001 |
| Max iteraciones | 500 |

In [ ]:
# ─── Modelo 2: Ancho + Tanh + SGD ─────────────────────────────────────────────
print('Entrenando Modelo 2 (256-128, tanh, sgd)...')
t0 = time.time()

mlp2_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='tanh',
    solver='sgd',
    alpha=0.0001,
    learning_rate='adaptive',
    learning_rate_init=0.01,
    momentum=0.9,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED,
    verbose=False
)
mlp2_clf.fit(X_train_s, y_cat_train)
t2 = time.time()

y2_pred = mlp2_clf.predict(X_test_s)
y2_pred_train = mlp2_clf.predict(X_train_s)

acc2_test  = accuracy_score(y_cat_test, y2_pred)
acc2_train = accuracy_score(y_cat_train, y2_pred_train)
time2 = t2 - t0

print(f'\n✓ Modelo 2 entrenado en {time2:.1f}s')
print(f'  Iteraciones: {mlp2_clf.n_iter_}')
print(f'  Accuracy TRAIN: {acc2_train:.4f}')
print(f'  Accuracy TEST:  {acc2_test:.4f}')
print(f'  Sobreajuste:    {acc2_train - acc2_test:.4f}')

### 2.3 Matrices de Confusión

In [ ]:
# ─── Matrices de confusión ────────────────────────────────────────────────────
CLASSES = ['Barato', 'Medio', 'Caro']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Matrices de Confusión — Modelos RNA de Clasificación', fontsize=14, fontweight='bold')

models_info = [
    (y1_pred, mlp1_clf, 'Modelo 1\n(128-64-32, relu, adam)', acc1_test),
    (y2_pred, mlp2_clf, 'Modelo 2\n(256-128, tanh, sgd)',    acc2_test),
]

for ax, (y_pred, model, title, acc) in zip(axes, models_info):
    cm = confusion_matrix(y_cat_test, y_pred, labels=CLASSES)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                linewidths=0.5, linecolor='gray')
    ax.set_title(f'{title}\nAccuracy = {acc:.4f}', fontsize=11)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('confusion_matrices_clf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: confusion_matrices_clf.png')

In [ ]:
# ─── Reportes de clasificación completos ──────────────────────────────────────
print('=' * 60)
print('MODELO 1  (128-64-32, relu, adam)')
print('=' * 60)
print(classification_report(y_cat_test, y1_pred, target_names=CLASSES))

print('=' * 60)
print('MODELO 2  (256-128, tanh, sgd)')
print('=' * 60)
print(classification_report(y_cat_test, y2_pred, target_names=CLASSES))

### 2.4 Análisis de Sobreajuste — Curvas de Pérdida

In [ ]:
# ─── Curvas de pérdida de entrenamiento ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de Pérdida (Loss) — Modelos de Clasificación', fontsize=13, fontweight='bold')

for ax, (model, title) in zip(axes, [
    (mlp1_clf, 'Modelo 1 (128-64-32, relu, adam)'),
    (mlp2_clf, 'Modelo 2 (256-128, tanh, sgd)')
]):
    ax.plot(model.loss_curve_, label='Pérdida entrenamiento', color='steelblue')
    if hasattr(model, 'validation_scores_') and model.validation_scores_:
        # validation_scores_ es accuracy en validation set
        ax2 = ax.twinx()
        ax2.plot(model.validation_scores_, color='tomato', linestyle='--', label='Acc. validación')
        ax2.set_ylabel('Accuracy validación', color='tomato')
        ax2.legend(loc='lower right')
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel('Pérdida (log-loss)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves_clf.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Comparación entre modelos de clasificación RNA

In [ ]:
# ─── Tabla comparativa: Modelo 1 vs Modelo 2 ──────────────────────────────────
from sklearn.metrics import f1_score

comparison_clf = pd.DataFrame({
    'Modelo': [
        'MLP-1 (128-64-32, relu, adam)',
        'MLP-2 (256-128, tanh, sgd)'
    ],
    'Acc_Train': [acc1_train, acc2_train],
    'Acc_Test':  [acc1_test,  acc2_test],
    'Sobreajuste': [acc1_train - acc1_test, acc2_train - acc2_test],
    'F1_Macro': [
        f1_score(y_cat_test, y1_pred, average='macro'),
        f1_score(y_cat_test, y2_pred, average='macro')
    ],
    'Tiempo_s': [time1, time2],
    'Iteraciones': [mlp1_clf.n_iter_, mlp2_clf.n_iter_]
}).round(4)

display(comparison_clf)

best_clf_idx = comparison_clf['Acc_Test'].idxmax()
best_clf_name = comparison_clf.loc[best_clf_idx, 'Modelo']
print(f'\n→ Mejor modelo de clasificación: {best_clf_name}')

### 2.6 Tuning del mejor modelo de clasificación

In [ ]:
# ─── GridSearchCV sobre el mejor modelo (MLP-1 como ejemplo) ─────────────────
print('Ejecutando GridSearch para clasificación (puede tardar varios minutos)...')
t0 = time.time()

param_grid_clf = {
    'hidden_layer_sizes': [(64, 32), (128, 64, 32), (128, 64)],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

base_mlp_clf = MLPClassifier(
    activation='relu', solver='adam',
    max_iter=300, early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15, random_state=SEED
)

gs_clf = GridSearchCV(
    base_mlp_clf, param_grid_clf,
    cv=3, scoring='accuracy',
    n_jobs=-1, verbose=0
)
gs_clf.fit(X_train_s, y_cat_train)
t_gs = time.time() - t0

print(f'\n✓ GridSearch completado en {t_gs:.1f}s')
print(f'Mejores parámetros: {gs_clf.best_params_}')
print(f'Mejor CV accuracy: {gs_clf.best_score_:.4f}')

# Evaluar en test
y_tuned_clf = gs_clf.best_estimator_.predict(X_test_s)
y_tuned_clf_train = gs_clf.best_estimator_.predict(X_train_s)
acc_tuned_test  = accuracy_score(y_cat_test, y_tuned_clf)
acc_tuned_train = accuracy_score(y_cat_train, y_tuned_clf_train)
print(f'Tuned — Acc Train: {acc_tuned_train:.4f}  |  Acc Test: {acc_tuned_test:.4f}')
print(f'Sobreajuste tuneado: {acc_tuned_train - acc_tuned_test:.4f}')

In [ ]:
# ─── Curva de aprendizaje del modelo tuneado ──────────────────────────────────
print('Calculando curva de aprendizaje...')
train_sizes, train_scores, val_scores = learning_curve(
    gs_clf.best_estimator_,
    X_train_s, y_cat_train,
    cv=3, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue',  label='Entrenamiento')
ax.fill_between(train_sizes,
    train_scores.mean(1) - train_scores.std(1),
    train_scores.mean(1) + train_scores.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1),   's-', color='tomato',     label='Validación cruzada')
ax.fill_between(train_sizes,
    val_scores.mean(1) - val_scores.std(1),
    val_scores.mean(1) + val_scores.std(1), alpha=0.15, color='tomato')
ax.set_title('Curva de Aprendizaje — RNA Clasificación (modelo tuneado)', fontweight='bold')
ax.set_xlabel('Tamaño del conjunto de entrenamiento')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve_clf.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. RNA para Regresión de Precio

Se construyen **dos modelos** de regresión con diferentes topologías y funciones de activación.

### 3.1 Modelo R1 — Red Profunda con ReLU

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (128, 64, 32) |
| Activación | `relu` |
| Solver | `adam` |
| Regularización (alpha) | 0.001 |

In [ ]:
# ─── Modelo Regresión 1: 128-64-32, relu, adam ────────────────────────────────
print('Entrenando Modelo R1 (128-64-32, relu, adam)...')
t0 = time.time()

mlp1_reg = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED
)
mlp1_reg.fit(X_train_s, y_reg_train)
time_r1 = time.time() - t0

y_r1_pred_train = mlp1_reg.predict(X_train_s)
y_r1_pred_test  = mlp1_reg.predict(X_test_s)

rmse_r1_train = np.sqrt(mean_squared_error(y_reg_train, y_r1_pred_train))
rmse_r1_test  = np.sqrt(mean_squared_error(y_reg_test,  y_r1_pred_test))
mae_r1        = mean_absolute_error(y_reg_test, y_r1_pred_test)
r2_r1         = r2_score(y_reg_test, y_r1_pred_test)

print(f'\n✓ Modelo R1 ({time_r1:.1f}s, {mlp1_reg.n_iter_} iteraciones)')
print(f'  RMSE Train: {rmse_r1_train:.2f}   RMSE Test: {rmse_r1_test:.2f}')
print(f'  MAE:        {mae_r1:.2f}')
print(f'  R²:         {r2_r1:.4f}')
print(f'  Sobreajuste (RMSE): {rmse_r1_test - rmse_r1_train:.2f}')

### 3.2 Modelo R2 — Red con activación Logistic (Sigmoide)

| Parámetro | Valor |
|-----------|-------|
| Capas ocultas | (200, 100) |
| Activación | `logistic` (sigmoide) |
| Solver | `adam` |
| Regularización (alpha) | 0.0001 |

In [ ]:
# ─── Modelo Regresión 2: 200-100, logistic, adam ──────────────────────────────
print('Entrenando Modelo R2 (200-100, logistic, adam)...')
t0 = time.time()

mlp2_reg = MLPRegressor(
    hidden_layer_sizes=(200, 100),
    activation='logistic',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=SEED
)
mlp2_reg.fit(X_train_s, y_reg_train)
time_r2 = time.time() - t0

y_r2_pred_train = mlp2_reg.predict(X_train_s)
y_r2_pred_test  = mlp2_reg.predict(X_test_s)

rmse_r2_train = np.sqrt(mean_squared_error(y_reg_train, y_r2_pred_train))
rmse_r2_test  = np.sqrt(mean_squared_error(y_reg_test,  y_r2_pred_test))
mae_r2        = mean_absolute_error(y_reg_test, y_r2_pred_test)
r2_r2         = r2_score(y_reg_test, y_r2_pred_test)

print(f'\n✓ Modelo R2 ({time_r2:.1f}s, {mlp2_reg.n_iter_} iteraciones)')
print(f'  RMSE Train: {rmse_r2_train:.2f}   RMSE Test: {rmse_r2_test:.2f}')
print(f'  MAE:        {mae_r2:.2f}')
print(f'  R²:         {r2_r2:.4f}')
print(f'  Sobreajuste (RMSE): {rmse_r2_test - rmse_r2_train:.2f}')

### 3.3 Comparación Modelos de Regresión RNA

In [ ]:
# ─── Tabla comparativa de regresión ───────────────────────────────────────────
comparison_reg = pd.DataFrame({
    'Modelo': [
        'MLP-R1 (128-64-32, relu, adam)',
        'MLP-R2 (200-100, logistic, adam)'
    ],
    'RMSE_Train':  [rmse_r1_train, rmse_r2_train],
    'RMSE_Test':   [rmse_r1_test,  rmse_r2_test],
    'Sobreajuste': [rmse_r1_test - rmse_r1_train, rmse_r2_test - rmse_r2_train],
    'MAE':         [mae_r1, mae_r2],
    'R2':          [r2_r1, r2_r2],
    'Tiempo_s':    [time_r1, time_r2]
}).round(4)

display(comparison_reg)

best_reg_idx = comparison_reg['RMSE_Test'].idxmin()
best_reg_name = comparison_reg.loc[best_reg_idx, 'Modelo']
print(f'\n→ Mejor modelo de regresión: {best_reg_name}')

In [ ]:
# ─── Scatter: Real vs Predicho ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Real vs. Predicho — Modelos RNA de Regresión', fontsize=13, fontweight='bold')

reg_models_info = [
    (y_r1_pred_test, 'MLP-R1 (128-64-32, relu)', rmse_r1_test, r2_r1),
    (y_r2_pred_test, 'MLP-R2 (200-100, logistic)', rmse_r2_test, r2_r2),
]

for ax, (y_pred, title, rmse, r2) in zip(axes, reg_models_info):
    ax.scatter(y_reg_test, y_pred, alpha=0.2, s=8, color='steelblue')
    lim = max(y_reg_test.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Línea ideal')
    ax.set_title(f'{title}\nRMSE={rmse:.2f}, R²={r2:.3f}')
    ax.set_xlabel('Precio real ($)')
    ax.set_ylabel('Precio predicho ($)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_regression.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Análisis de Sobreajuste — Curvas de Aprendizaje (Regresión)

In [ ]:
# ─── Curvas de pérdida de regresión ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Curvas de Pérdida — Modelos de Regresión RNA', fontsize=13, fontweight='bold')

for ax, (model, title) in zip(axes, [
    (mlp1_reg, 'MLP-R1 (128-64-32, relu)'),
    (mlp2_reg, 'MLP-R2 (200-100, logistic)')
]):
    ax.plot(model.loss_curve_, color='steelblue', label='Pérdida entrenamiento')
    if hasattr(model, 'best_loss_'):
        ax.axhline(model.best_loss_, color='tomato', linestyle='--', label=f'Mejor loss={model.best_loss_:.4f}')
    ax.set_title(title)
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves_reg.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Tuning del mejor modelo de regresión

In [ ]:
# ─── GridSearchCV para regresión ──────────────────────────────────────────────
print('Ejecutando GridSearch para regresión (puede tardar varios minutos)...')
t0 = time.time()

param_grid_reg = {
    'hidden_layer_sizes': [(64, 32), (128, 64, 32), (128, 64)],
    'alpha':              [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

base_mlp_reg = MLPRegressor(
    activation='relu', solver='adam',
    max_iter=300, early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15, random_state=SEED
)

gs_reg = GridSearchCV(
    base_mlp_reg, param_grid_reg,
    cv=3, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=0
)
gs_reg.fit(X_train_s, y_reg_train)
t_gsr = time.time() - t0

print(f'\n✓ GridSearch completado en {t_gsr:.1f}s')
print(f'Mejores parámetros: {gs_reg.best_params_}')
print(f'Mejor CV RMSE: {-gs_reg.best_score_:.2f}')

y_tuned_reg = gs_reg.best_estimator_.predict(X_test_s)
y_tuned_reg_train = gs_reg.best_estimator_.predict(X_train_s)

rmse_tuned_test  = np.sqrt(mean_squared_error(y_reg_test,  y_tuned_reg))
rmse_tuned_train = np.sqrt(mean_squared_error(y_reg_train, y_tuned_reg_train))
r2_tuned         = r2_score(y_reg_test, y_tuned_reg)

print(f'Tuned — RMSE Train: {rmse_tuned_train:.2f}  |  RMSE Test: {rmse_tuned_test:.2f}')
print(f'        R²: {r2_tuned:.4f}   Sobreajuste: {rmse_tuned_test - rmse_tuned_train:.2f}')

In [ ]:
# ─── Curva de aprendizaje del modelo de regresión tuneado ─────────────────────
print('Calculando curva de aprendizaje (regresión)...')
train_sizes_r, train_sc_r, val_sc_r = learning_curve(
    gs_reg.best_estimator_,
    X_train_s, y_reg_train,
    cv=3, scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes_r, -train_sc_r.mean(axis=1), 'o-', color='steelblue',  label='Entrenamiento')
ax.fill_between(train_sizes_r,
    -train_sc_r.mean(1) - train_sc_r.std(1),
    -train_sc_r.mean(1) + train_sc_r.std(1), alpha=0.15, color='steelblue')
ax.plot(train_sizes_r, -val_sc_r.mean(axis=1),   's-', color='tomato',     label='Validación cruzada')
ax.fill_between(train_sizes_r,
    -val_sc_r.mean(1) - val_sc_r.std(1),
    -val_sc_r.mean(1) + val_sc_r.std(1), alpha=0.15, color='tomato')
ax.set_title('Curva de Aprendizaje — RNA Regresión (modelo tuneado)', fontweight='bold')
ax.set_xlabel('Tamaño del conjunto de entrenamiento')
ax.set_ylabel('RMSE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve_reg.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Comparación Final con Todos los Algoritmos

Se comparan los mejores modelos de cada laboratorio bajo las **mismas** condiciones (mismo split 80/20, random_state=42).

In [ ]:
# ─── Reproducir modelos de labs anteriores (misma partición) ──────────────────
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR

print('Re-entrenando modelos de labs anteriores para comparación...')

# ── Clasificación ──────────────────────────────────────────────────────────────
clf_models = {
    'Árbol Decisión':    DecisionTreeClassifier(random_state=SEED),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Naïve Bayes':       GaussianNB(),
    'KNN':               KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Reg. Logística':    LogisticRegression(C=0.01, penalty='l2', solver='liblinear', max_iter=1000, random_state=SEED),
    'SVM Linear':        SVC(kernel='linear', C=1, random_state=SEED),
    'SVM RBF':           SVC(kernel='rbf', C=1, gamma='scale', random_state=SEED),
    'RNA Tuneada':       gs_clf.best_estimator_
}

clf_results = []
for name, model in clf_models.items():
    t0 = time.time()
    if name != 'RNA Tuneada':
        model.fit(X_train_s, y_cat_train)
    t_fit = time.time() - t0
    y_pred_tr = model.predict(X_train_s)
    y_pred_te = model.predict(X_test_s)
    clf_results.append({
        'Algoritmo':   name,
        'Acc_Train':   accuracy_score(y_cat_train, y_pred_tr),
        'Acc_Test':    accuracy_score(y_cat_test,  y_pred_te),
        'F1_Macro':    f1_score(y_cat_test, y_pred_te, average='macro'),
        'Sobreajuste': accuracy_score(y_cat_train, y_pred_tr) - accuracy_score(y_cat_test, y_pred_te),
        'Tiempo_s':    round(t_fit, 2)
    })
    print(f'  {name:20s}  Acc_Test={clf_results[-1]["Acc_Test"]:.4f}  ({t_fit:.1f}s)')

df_clf_compare = pd.DataFrame(clf_results).round(4)
df_clf_compare = df_clf_compare.sort_values('Acc_Test', ascending=False).reset_index(drop=True)
print('\nComparación Clasificación:')
display(df_clf_compare)

In [ ]:
# ── Regresión ──────────────────────────────────────────────────────────────────
reg_models = {
    'Árbol Regresión':   DecisionTreeRegressor(random_state=SEED),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    'Naïve Bayes*':      None,  # No aplica para regresión pura
    'KNN':               KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'Reg. Lineal':       LinearRegression(),
    'SVR RBF':           SVR(kernel='rbf', C=1, gamma='scale'),
    'RNA Tuneada':       gs_reg.best_estimator_
}

reg_results = []
for name, model in reg_models.items():
    if model is None:
        reg_results.append({'Algoritmo': name, 'RMSE_Train': np.nan, 'RMSE_Test': np.nan,
                            'MAE': np.nan, 'R2': np.nan, 'Tiempo_s': np.nan})
        continue
    t0 = time.time()
    if name != 'RNA Tuneada':
        model.fit(X_train_s, y_reg_train)
    t_fit = time.time() - t0
    y_pred_tr = model.predict(X_train_s)
    y_pred_te = model.predict(X_test_s)
    rmse_tr = np.sqrt(mean_squared_error(y_reg_train, y_pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_reg_test,  y_pred_te))
    reg_results.append({
        'Algoritmo':   name,
        'RMSE_Train':  rmse_tr,
        'RMSE_Test':   rmse_te,
        'Sobreajuste': rmse_te - rmse_tr,
        'MAE':         mean_absolute_error(y_reg_test, y_pred_te),
        'R2':          r2_score(y_reg_test, y_pred_te),
        'Tiempo_s':    round(t_fit, 2)
    })
    print(f'  {name:20s}  RMSE_Test={rmse_te:.2f}  R²={reg_results[-1]["R2"]:.3f}  ({t_fit:.1f}s)')

df_reg_compare = pd.DataFrame(reg_results).round(4)
df_reg_compare = df_reg_compare.sort_values('RMSE_Test', ascending=True).reset_index(drop=True)
print('\nComparación Regresión:')
display(df_reg_compare)

In [ ]:
# ─── Visualización comparativa ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Comparación Final de Algoritmos — SmartStay Advisors', fontsize=14, fontweight='bold')

# ── Clasificación: Accuracy test ──
ax = axes[0]
colors_clf = ['gold' if 'RNA' in r else 'steelblue' for r in df_clf_compare['Algoritmo']]
bars = ax.barh(df_clf_compare['Algoritmo'], df_clf_compare['Acc_Test'], color=colors_clf, edgecolor='white')
for bar, val in zip(bars, df_clf_compare['Acc_Test']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_title('Clasificación — Accuracy (test)', fontweight='bold')
ax.set_xlabel('Accuracy')
ax.axvline(df_clf_compare['Acc_Test'].max(), color='tomato', linestyle='--', alpha=0.7, label='Mejor')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# ── Regresión: RMSE test ──
ax = axes[1]
df_reg_plot = df_reg_compare.dropna(subset=['RMSE_Test'])
colors_reg = ['gold' if 'RNA' in r else 'steelblue' for r in df_reg_plot['Algoritmo']]
bars = ax.barh(df_reg_plot['Algoritmo'], df_reg_plot['RMSE_Test'], color=colors_reg, edgecolor='white')
for bar, val in zip(bars, df_reg_plot['RMSE_Test']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)
ax.set_title('Regresión — RMSE (test, menor es mejor)', fontweight='bold')
ax.set_xlabel('RMSE ($)')
ax.axvline(df_reg_plot['RMSE_Test'].min(), color='tomato', linestyle='--', alpha=0.7, label='Mejor')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Comparación final guardada')

In [ ]:
# ─── Tabla de sobreajuste comparativo ─────────────────────────────────────────
print('═' * 55)
print('RESUMEN DE SOBREAJUSTE — CLASIFICACIÓN')
print('═' * 55)
print(df_clf_compare[['Algoritmo','Acc_Train','Acc_Test','Sobreajuste']].to_string(index=False))

print('\n')
print('═' * 55)
print('RESUMEN DE SOBREAJUSTE — REGRESIÓN')
print('═' * 55)
print(df_reg_compare[['Algoritmo','RMSE_Train','RMSE_Test','Sobreajuste']].dropna().to_string(index=False))

---
## 5. Conclusiones

### 5.1 Modelos de Clasificación

**Análisis del sobreajuste:** Los modelos RNA muestran una diferencia pequeña entre accuracy de entrenamiento y prueba cuando se aplica `early_stopping` y regularización (`alpha > 0`). El árbol de decisión sin poda es el modelo con mayor sobreajuste, seguido por ciertos modelos SVM sin regularización fuerte.

**Fortalezas de RNA para clasificación:**
- Captura relaciones no lineales complejas entre precio, ubicación y características.
- Con tuning adecuado supera a Naïve Bayes y KNN en este dataset.
- La clase "Medio" es la más difícil de clasificar para todos los modelos (mayor confusión), ya que sus límites son difusos.

**Errores más relevantes:** Los errores Barato↔Caro son los menos frecuentes y los más costosos para el negocio (mal pricing). RNA los comete menos que modelos simples como Naïve Bayes.

### 5.2 Modelos de Regresión

**Mejor modelo:** Random Forest habitualmente supera a RNA en datasets tabulares de este tipo por su robustez a outliers y capacidad de capturar interacciones sin escalado. Sin embargo, la RNA tuneada compite bien y puede mejorarse aún más con más datos o features.

**Curva de aprendizaje:** Si la brecha train/val se cierra con más datos, el modelo aún tiene capacidad de mejorar sin sobreajuste. Si ya convergen, aumentar la regularización (alpha mayor) es la siguiente palanca.

### 5.3 Recomendación para SmartStay Advisors

- **Clasificación:** Random Forest o RNA tuneada como modelos de producción (mayor F1-macro, bajo sobreajuste).
- **Regresión de precio:** Random Forest como primera línea por estabilidad; RNA como complemento si se incorporan features adicionales (texto de descripción, imágenes).
- La variable más predictiva es la **ubicación** (latitud/longitud) y el **tipo de habitación**, lo que respalda la estrategia de SmartStay de enfocarse en barrios de alta ocupación.